# Phase 1 — Data Setup & Verification

**Project:** Sleep Doctor — predicting sleep quality from age, stress, and activity.

**Goal of this notebook:** load the Sleep Health dataset, verify its integrity (missing values, duplicates, impossible values), and document why no cleaning was needed. Every later notebook builds on the checks done here.

**Dataset:** `sleep_health_dataset.csv` — 100,000 records from [Kaggle](https://www.kaggle.com/datasets/mohankrishnathalla/sleep-health-and-daily-performance-dataset). Note: this is a **synthetic** dataset, so our conclusions describe the patterns built into it, not proven facts about real human sleep.

## 1. Load the dataset

The notebook lives in `notebooks/`, and the CSV sits one level up — hence the `../` in the path. `df.shape` returns `(rows, columns)`, our first integrity check: we expect exactly **100,000 rows and 32 columns**.

In [1]:
import pandas as pd

df = pd.read_csv("../sleep_health_dataset.csv")
df.shape

(100000, 32)

## 2. First look at the data

`df.head()` shows the first five rows. The point is to understand **what one row represents**: one person's day — their demographics (age, gender, occupation, country), lifestyle that day (steps, exercise, caffeine, screen time, work hours), and how they slept (duration, quality score, REM %, wake episodes...).

In [2]:
df.head()

,person_id,age,gender,occupation,bmi,country,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,...,heart_rate_resting_bpm,sleep_aid_used,shift_work,room_temperature_celsius,weekend_sleep_diff_hrs,season,day_type,cognitive_performance_score,sleep_disorder_risk,felt_rested
0,1,29,Female,Driver,25.7,Japan,6.19,6.6,22.5,19.3,...,63,0,0,20.1,1.84,Autumn,Weekday,73.4,Healthy,0
1,2,55,Female,Software Engineer,22.0,USA,8.32,6.9,26.9,14.9,...,52,1,0,18.0,0.13,Winter,Weekend,99.4,Healthy,1
2,3,42,Male,Nurse,25.0,India,3.74,1.0,20.2,16.2,...,72,0,1,17.9,1.67,Spring,Weekend,2.5,Severe,0
3,4,37,Female,Student,29.5,India,6.79,6.4,17.7,17.7,...,71,0,0,19.1,2.37,Summer,Weekend,67.8,Healthy,0
4,5,23,Male,Lawyer,23.6,Spain,5.02,3.2,23.3,18.3,...,71,0,0,19.7,1.26,Summer,Weekday,38.1,Mild,0


## 3. Column types

`df.info()` lists every column with its dtype. What we're checking: **numeric columns should be `int64`/`float64`, text columns `object`.** If a numeric column showed up as `object`, that would mean dirty values are hiding in it (like a stray `"N/A"` string) — that's exactly how this check catches problems.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 32 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   person_id                    100000 non-null  int64  
 1   age                          100000 non-null  int64  
 2   gender                       100000 non-null  object 
 3   occupation                   100000 non-null  object 
 4   bmi                          100000 non-null  float64
 5   country                      100000 non-null  object 
 6   sleep_duration_hrs           100000 non-null  float64
 7   sleep_quality_score          100000 non-null  float64
 8   rem_percentage               100000 non-null  float64
 9   deep_sleep_percentage        100000 non-null  float64
 10  sleep_latency_mins           100000 non-null  int64  
 11  wake_episodes_per_night      100000 non-null  int64  
 12  caffeine_mg_before_bed       100000 non-null  int64  
 13  

## 4. Missing values

`isnull().sum()` counts missing entries per column. If we found any, we'd have to decide between **deletion** (dropping rows/columns) and **imputation** (filling with mean/median/mode — ideally per-group, so we don't erase differences between groups). We expect **all zeros** here, which means: nothing to impute, and we can *prove* it rather than assume it.

In [4]:
df.isnull().sum()

person_id                      0
age                            0
gender                         0
occupation                     0
bmi                            0
country                        0
sleep_duration_hrs             0
sleep_quality_score            0
rem_percentage                 0
deep_sleep_percentage          0
sleep_latency_mins             0
wake_episodes_per_night        0
caffeine_mg_before_bed         0
alcohol_units_before_bed       0
screen_time_before_bed_mins    0
exercise_day                   0
steps_that_day                 0
nap_duration_mins              0
stress_score                   0
work_hours_that_day            0
chronotype                     0
mental_health_condition        0
heart_rate_resting_bpm         0
sleep_aid_used                 0
shift_work                     0
room_temperature_celsius       0
weekend_sleep_diff_hrs         0
season                         0
day_type                       0
cognitive_performance_score    0
sleep_diso

## 5. Duplicates

Two checks: fully identical rows, and duplicated `person_id`s (same person appearing twice would silently double-weight them in training). We expect **0 and 0** — confirming each row is one unique person.

In [5]:
print("duplicate rows:      ", df.duplicated().sum())
print("duplicate person_ids:", df["person_id"].duplicated().sum())

duplicate rows:       0
duplicate person_ids: 0


## 6. Sanity-check value ranges

A dataset can have zero missing values and still contain **impossible** ones (age 250, negative steps). `describe()` gives min/max/quartiles for the columns our research question uses — we check each against domain knowledge:

| Column | Plausible range | Why |
|---|---|---|
| `age` | 18–69 | adult study population |
| `stress_score` | 1–10 | defined scale |
| `steps_that_day` | 500–20,000 | sedentary day → very active day |
| `sleep_duration_hrs` | 3–10.5 | short night → long night |
| `sleep_quality_score` | 1–10 | defined scale (our target) |

In [6]:
df[["age", "stress_score", "steps_that_day", "sleep_duration_hrs", "sleep_quality_score"]].describe()

,age,stress_score,steps_that_day,sleep_duration_hrs,sleep_quality_score
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,34.706870,5.733285,7496.859740,6.423986,4.871144
std,11.036373,1.619194,3460.423881,1.274627,1.506517
min,18.000000,1.000000,500.000000,3.000000,1.000000
25%,26.000000,4.800000,5045.000000,5.530000,3.800000
50%,33.000000,5.800000,7442.000000,6.360000,4.900000
75%,42.000000,6.800000,9887.000000,7.270000,6.000000
max,69.000000,10.000000,20000.000000,10.500000,10.000000


## 7. Findings

- The dataset loaded with **100,000 rows and 32 columns**, exactly as documented.
- **No missing values** in any column, and **no duplicate rows or person IDs** — so no imputation or deduplication was needed.
- All value ranges are **humanly plausible** (age 18–69, stress and sleep quality within their 1–10 scales, steps 500–20,000, sleep duration 3–10.5 hrs), so no rows were dropped as impossible.
- **Ethics note:** rather than automatically deleting statistical outliers, we checked extremes against domain knowledge and kept them. Blind outlier removal can silently erase real groups of people (e.g., shift workers with very short sleep), which would bias every model trained afterward.

**Bottom line:** the data is clean and ready. Phase 2 (statistics) can work directly on this dataframe with no preprocessing.